# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a Croissant-encoded dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema, available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The FAIR² dataset contains regression outputs, socio-demographics, and survey response data for rangeland management studies in Northern Kenya.

---

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and available records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"Dataset title: {meta.name}\n\nDescription: {meta.description}\n")

## 2. Data Overview

Identify available record sets, their fields, and list all IDs. For reproducibility and clarity, all references are made by the object's `@id`.

Fields within each record set (such as regression results or survey responses) are also listed by their `@id`.

In [ ]:
# List all record sets in the dataset

print("Available record sets and field IDs in the dataset:")
record_sets = []
for rset in dataset.record_sets:
    print(f"- Record set @id: {rset.id}, name: {rset.name}")
    print("    Fields/columns:")
    for fld in rset.fields:
        print(f"      - Field @id: {fld.id}, name: {fld.name}")
    record_sets.append(rset.id)
print("\nTotal Record Sets Found:", len(record_sets))

## 3. Data Extraction

Load the records from each identified record set into pandas DataFrames for analysis. Use record set and field `@id`s for precise data access.

In [ ]:
# Prepare DataFrames for all record sets
# You can adapt indexes below based on the printed record_set ids above
# For demonstration we extract the first available record set (if there is one)

dataframes = {}
if len(record_sets) == 0:
    print("No record sets found in dataset. Please check the dataset or Croissant schema.")
else:
    for record_set_id in record_sets:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}.")
    # Preview columns of first record set
    primary_set = record_sets[0]
    print("\nPrimary record set columns (by @id):")
    print(dataframes[primary_set].columns.tolist())
    dataframes[primary_set].head()

## 4. Exploratory Data Analysis (EDA)

Perform common data processing steps:
- Filter DataFrame rows by numeric field values
- Normalize numeric fields
- Optionally group by a categorical field

All field references are made by their field `@id` (e.g., `'http://mlcommons.org/croissant/field/age'`). Edit the IDs below as discovered above.

In [ ]:
# Define which record set and field(s) to use for EDA
# Please set these to actual IDs printed above

# Example: using the first record set found
record_set_id = record_sets[0] if len(record_sets) > 0 else None

df = dataframes[record_set_id] if record_set_id is not None else pd.DataFrame()
print(f"Using record set: {record_set_id}\nNumber of records: {len(df)}")
print("Available columns:", df.columns.tolist())

# Choose a numeric field by @id (edit as appropriate)
import numpy as np
numeric_field_id = None

# Try to automatically select a likely numeric column
for col in df.columns:
    if df[col].dtype in [np.float64, np.int64, float, int]:
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field found in this record set.")
else:
    print(f"\nNumeric field selected for filtering: {numeric_field_id}\n")
    # Set an arbitrary threshold for demonstration
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered rows where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    )
    print(f"\nNormalized {numeric_field_id} column:")
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try to automatically group by a likely categorical column (string/object type with a reasonable number of values)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < len(df) // 2:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_" + numeric_field_id)
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found to group by.")

## 5. Visualization

Visualize the distribution of the numeric variable and show grouped means (if groups exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field_id:
        plt.figure(figsize=(10,4))
        grouped_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=grouped_means.index, y=grouped_means.values, palette='deep')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated loading a Croissant-formatted, FAIR-compliant dataset with `mlcroissant`.
- Record sets and fields were discovered *by their `@id`* for robust access.
- Basic filtering, normalization, grouping, and visualization of a numeric variable were shown.

**To go further**: integrate domain-specific analysis, examine variable relationships, and consult the Croissant schema for fully FAIR-aligned workflows.